# Phase 1 — Data Cleaning
**Étapes 1 & 2 : Chargement, Encodage, Renommage, Valeurs Manquantes**

| Paramètre | Valeur détectée |
|---|---|
| Encodage | `utf-8` (bytes `0xC3 0xA9` = é UTF-8 — pas latin-1) |
| Séparateur | `,` (pas `;` comme supposé initialement) |
| Colonne segment | `Ségment stratégique` (accent sur S) |

In [2]:
import pandas as pd
import numpy as np
import os

BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
print('Base dir:', BASE_DIR)

Base dir: c:\Users\lenovo\Desktop\Extraction livraison client 2021-2025


---
# Étape 1 — Chargement, Correction Encodage, Renommage

## 1.1 — Chargement avec le bon encodage

In [3]:
# Encoding = utf-8 (confirmé par analyse octets bruts : 0xC3 0xA9 = é UTF-8)
# Séparateur = ',' (confirmé par inspection)
df_cmd = pd.read_csv(
    os.path.join(BASE_DIR, 'Extraction commande client 2021-2025.csv'),
    encoding='utf-8',
    sep=','
)
df_liv = pd.read_csv(
    os.path.join(BASE_DIR, 'Extraction livraison client 2021-2025.csv'),
    encoding='utf-8',
    sep=','
)
print(f'Commandes  : {df_cmd.shape[0]:,} lignes x {df_cmd.shape[1]} colonnes')
print(f'Livraisons : {df_liv.shape[0]:,} lignes x {df_liv.shape[1]} colonnes')

Commandes  : 352,549 lignes x 14 colonnes
Livraisons : 353,325 lignes x 13 colonnes


In [4]:
print('Colonnes commandes :')
for c in df_cmd.columns: print(f'  {repr(c)}')
print()
print('Colonnes livraisons :')
for c in df_liv.columns: print(f'  {repr(c)}')

Colonnes commandes :
  '#'
  'Num commande'
  "Numéro d'article"
  'Quantité'
  "Date d'enregistrement"
  'Date de livraison demandée'
  'Code client/fournisseur'
  'Activité stratégique client'
  'Activité stratégique article'
  'Ségment stratégique'
  'Type activité'
  'Pays/région'
  'Prix de vente'
  'Devise du prix'

Colonnes livraisons :
  '#'
  'Num commande'
  "Numéro d'article"
  'Qté'
  'Date de livraison réelle'
  'Code client/fournisseur'
  'Activité stratégique client'
  'Activité stratégique article'
  'Ségment stratégique'
  'Type activité'
  'Pays/région'
  'Prix de vente'
  'Devise du prix'


In [5]:
df_cmd.head(3)

,#,Num commande,Numéro d'article,Quantité,Date d'enregistrement,Date de livraison demandée,Code client/fournisseur,Activité stratégique client,Activité stratégique article,Ségment stratégique,Type activité,Pays/région,Prix de vente,Devise du prix
0,1,142660,A0276,300,23/09/2020,15/01/2021,C10379,ELEVAGE,ELEVAGE,ECORNAGE,Outil GAZ,IE,89.00,EUR
1,2,143298,A1577,2,08/10/2020,04/01/2021,C10567,CONSTRUCTION,CONSTRUCTION,COUVERTURE,CONSOMMABLE,FR,9.04,EUR
2,3,143298,A1571,1,08/10/2020,04/01/2021,C10567,CONSTRUCTION,CONSTRUCTION,OUTIL CHAUFFANT,PIECES DETACHEES,FR,11.45,EUR


In [6]:
df_liv.head(3)

,#,Num commande,Numéro d'article,Qté,Date de livraison réelle,Code client/fournisseur,Activité stratégique client,Activité stratégique article,Ségment stratégique,Type activité,Pays/région,Prix de vente,Devise du prix
0,1,143298,A1577,2,04/01/2021,C10567,CONSTRUCTION,CONSTRUCTION,COUVERTURE,CONSOMMABLE,FR,9.04,EUR
1,2,143298,A1571,1,04/01/2021,C10567,CONSTRUCTION,CONSTRUCTION,OUTIL CHAUFFANT,PIECES DETACHEES,FR,11.45,EUR
2,3,143298,A1283,1,04/01/2021,C10567,CONSTRUCTION,CONSTRUCTION,COUVERTURE,CONSOMMABLE,FR,17.19,EUR


## 1.2 — Renommage des colonnes (snake_case)

In [7]:
# Note : colonne réelle = 'Ségment stratégique' (accent sur S)
rename_cmd = {
    '#':                                'row_id',
    'Num commande':                     'num_commande',
    "Numéro d'article":                 'code_article',
    'Quantité':                         'qte_demandee',
    "Date d'enregistrement":            'date_enregistrement',
    'Date de livraison demandée':       'date_livraison_demandee',
    'Code client/fournisseur':          'code_client',
    'Activité stratégique client':      'famille_activite_client',
    'Activité stratégique article':     'famille_activite_article',
    'Ségment stratégique':              'segment',
    'Type activité':                    'type_activite',
    'Pays/région':                      'pays',
    'Prix de vente':                    'prix',
    'Devise du prix':                   'devise',
}
df_cmd = df_cmd.rename(columns=rename_cmd)
print('Colonnes CMD:', list(df_cmd.columns))

Colonnes CMD: ['row_id', 'num_commande', 'code_article', 'qte_demandee', 'date_enregistrement', 'date_livraison_demandee', 'code_client', 'famille_activite_client', 'famille_activite_article', 'segment', 'type_activite', 'pays', 'prix', 'devise']


In [8]:
rename_liv = {
    '#':                                'row_id',
    'Num commande':                     'num_commande',
    "Numéro d'article":                 'code_article',
    'Qté':                              'qte_livree',
    'Date de livraison réelle':         'date_livraison_reelle',
    'Code client/fournisseur':          'code_client',
    'Activité stratégique client':      'famille_activite_client',
    'Activité stratégique article':     'famille_activite_article',
    'Ségment stratégique':              'segment',
    'Type activité':                    'type_activite',
    'Pays/région':                      'pays',
    'Prix de vente':                    'prix',
    'Devise du prix':                   'devise',
}
df_liv = df_liv.rename(columns=rename_liv)
print('Colonnes LIV:', list(df_liv.columns))

Colonnes LIV: ['row_id', 'num_commande', 'code_article', 'qte_livree', 'date_livraison_reelle', 'code_client', 'famille_activite_client', 'famille_activite_article', 'segment', 'type_activite', 'pays', 'prix', 'devise']


## 1.3 — Vérification rapide post-chargement

In [9]:
print('=== COMMANDES ===')
print(f'Shape : {df_cmd.shape}')
print()
print('Valeurs nulles :')
print(df_cmd.isnull().sum())
print()
print('Types :')
print(df_cmd.dtypes)

=== COMMANDES ===
Shape : (352549, 14)

Valeurs nulles :
row_id                         0
num_commande                   0
code_article                  31
qte_demandee                   0
date_enregistrement            0
date_livraison_demandee        0
code_client                    0
famille_activite_client       17
famille_activite_article       0
segment                       19
type_activite                  0
pays                          99
prix                           0
devise                      1987
dtype: int64

Types :
row_id                        int64
num_commande                  int64
code_article                    str
qte_demandee                  int64
date_enregistrement             str
date_livraison_demandee         str
code_client                     str
famille_activite_client         str
famille_activite_article        str
segment                         str
type_activite                   str
pays                            str
prix                       

In [10]:
print('=== LIVRAISONS ===')
print(f'Shape : {df_liv.shape}')
print()
print('Valeurs nulles :')
print(df_liv.isnull().sum())
print()
print('Types :')
print(df_liv.dtypes)

=== LIVRAISONS ===
Shape : (353325, 13)

Valeurs nulles :


row_id                         0
num_commande                   0
code_article                  30
qte_livree                     0
date_livraison_reelle          0
code_client                    0
famille_activite_client       17
famille_activite_article       0
segment                       19
type_activite                  0
pays                         100
prix                           0
devise                      1977
dtype: int64

Types :
row_id                        int64
num_commande                  int64
code_article                    str
qte_livree                    int64
date_livraison_reelle           str
code_client                     str
famille_activite_client         str
famille_activite_article        str
segment                         str
type_activite                   str
pays                            str
prix                        float64
devise                          str
dtype: object


In [11]:
# Doublons num_commande (attendu : 1 commande = N articles)
print('Num commandes uniques CMD :', df_cmd['num_commande'].nunique())
print('Num commandes uniques LIV :', df_liv['num_commande'].nunique())
print()
print('Doublons num_commande CMD :', df_cmd.duplicated(subset='num_commande').sum())
print('Doublons num_commande LIV :', df_liv.duplicated(subset='num_commande').sum())

Num commandes uniques CMD : 62263
Num commandes uniques LIV : 62242

Doublons num_commande CMD : 290286
Doublons num_commande LIV : 291083


## 1.4 — Sauvegarde intermédiaire step1

In [12]:
out_dir = os.path.join(BASE_DIR, 'data', 'processed')
os.makedirs(out_dir, exist_ok=True)
df_cmd.to_csv(os.path.join(out_dir, 'cmd_step1.csv'), index=False, encoding='utf-8')
df_liv.to_csv(os.path.join(out_dir, 'liv_step1.csv'), index=False, encoding='utf-8')
print('Sauvegarde OK :', os.listdir(out_dir))

Sauvegarde OK : ['cmd_step1.csv', 'cmd_step2.csv', 'cmd_step3.csv', 'dataset_step4.csv', 'liv_step1.csv', 'liv_step2.csv', 'liv_step3.csv']


---
# Étape 2 — Traitement des Valeurs Manquantes

| Colonne | CMD | LIV | Stratégie |
|---|---|---|---|
| `code_article` | 31 | 30 | Suppression (article inconnu = ligne inutilisable) |
| `segment` | 19 | 19 | Résolu par suppression code_article (100% overlap) |
| `pays` | 99 | 100 | Mode par `code_client`, sinon `UNKNOWN` |
| `devise` | ~1987 | ~1977 | Mode par `pays`, sinon `EUR` |
| `famille_activite_client` | 17 | 17 | Mode par `code_client`, sinon `AUTRE` |

## 2.1 — code_article : suppression des lignes null uniquement

Le modèle prédit **au niveau ligne** (`code_client × code_article`).  
Les autres lignes d'une même commande sont des données valides et indépendantes → on les garde.

Décision : **supprimer seulement les 31/30 lignes sans article** (0,009% des données).

In [13]:
# Aperçu des lignes concernées
print('CMD : lignes sans code_article')
print(df_cmd[df_cmd['code_article'].isna()][['num_commande','qte_demandee','prix','pays']].head(5).to_string())

CMD : lignes sans code_article
       num_commande  qte_demandee   prix pays
1504         146051           800   0.09   US
2881         146243            95   6.00   US
42422        152567            15  25.68   US
42982        152643             5   0.00   FR
42983        152643             5   0.00   FR


In [14]:
n_cmd, n_liv = len(df_cmd), len(df_liv)

# Supprimer seulement les lignes avec code_article null
df_cmd = df_cmd.dropna(subset=['code_article'])
df_liv = df_liv.dropna(subset=['code_article'])

print(f'CMD : {n_cmd:,} -> {len(df_cmd):,} ({n_cmd - len(df_cmd)} lignes supprimées)')
print(f'LIV : {n_liv:,} -> {len(df_liv):,} ({n_liv - len(df_liv)} lignes supprimées)')
print(f'Nulls code_article restants : {df_cmd["code_article"].isna().sum()} / {df_liv["code_article"].isna().sum()}')

CMD : 352,549 -> 352,518 (31 lignes supprimées)
LIV : 353,325 -> 353,295 (30 lignes supprimées)
Nulls code_article restants : 0 / 0


## 2.2 — pays : imputation par mode du code_client

18/19 clients avec `pays` null ont d'autres commandes avec pays connu.  
1 client (C12663) n'a que 2 commandes, toutes sans pays → `UNKNOWN`.

In [15]:
def impute_pays(df):
    mode_pays = (
        df.dropna(subset=['pays'])
          .groupby('code_client')['pays']
          .agg(lambda x: x.mode()[0] if len(x) > 0 else None)
    )
    n = df['pays'].isna().sum()
    df['pays'] = df.apply(
        lambda row: mode_pays.get(row['code_client'], 'UNKNOWN') if pd.isna(row['pays']) else row['pays'],
        axis=1
    )
    df['pays'] = df['pays'].fillna('UNKNOWN')
    print(f'  pays null : {n} -> {df["pays"].isna().sum()} (mode client ou UNKNOWN)')
    return df

df_cmd = impute_pays(df_cmd)
df_liv = impute_pays(df_liv)

  pays null : 99 -> 0 (mode client ou UNKNOWN)
  pays null : 100 -> 0 (mode client ou UNKNOWN)


## 2.3 — devise : imputation par devise majoritaire du pays

Forte corrélation pays → devise : FR=EUR, CA=USD, CN=USD, GB=EUR, AU=EUR, US=USD.

In [16]:
def impute_devise(df):
    devise_maj = (
        df.dropna(subset=['devise'])
          .groupby('pays')['devise']
          .agg(lambda x: x.mode()[0] if len(x) > 0 else 'EUR')
    )
    n = df['devise'].isna().sum()
    df['devise'] = df.apply(
        lambda row: devise_maj.get(row['pays'], 'EUR') if pd.isna(row['devise']) else row['devise'],
        axis=1
    )
    df['devise'] = df['devise'].fillna('EUR')
    print(f'  devise null : {n} -> {df["devise"].isna().sum()} (mode pays ou EUR)')
    return df

df_cmd = impute_devise(df_cmd)
df_liv = impute_devise(df_liv)

  devise null : 1981 -> 0 (mode pays ou EUR)
  devise null : 1972 -> 0 (mode pays ou EUR)


## 2.4 — famille_activite_client : imputation par mode du client

In [17]:
def impute_famille(df):
    mode_fam = (
        df.dropna(subset=['famille_activite_client'])
          .groupby('code_client')['famille_activite_client']
          .agg(lambda x: x.mode()[0] if len(x) > 0 else None)
    )
    n = df['famille_activite_client'].isna().sum()
    df['famille_activite_client'] = df.apply(
        lambda row: mode_fam.get(row['code_client'], 'AUTRE') if pd.isna(row['famille_activite_client']) else row['famille_activite_client'],
        axis=1
    )
    df['famille_activite_client'] = df['famille_activite_client'].fillna('AUTRE')
    print(f'  famille_activite_client null : {n} -> {df["famille_activite_client"].isna().sum()}')
    return df

df_cmd = impute_famille(df_cmd)
df_liv = impute_famille(df_liv)

  famille_activite_client null : 17 -> 0
  famille_activite_client null : 17 -> 0


## 2.5 — Vérification finale et sauvegarde

In [18]:
print('=== RESULTAT ETAPE 2 ===')
print(f'CMD : {df_cmd.shape}')
print(f'LIV : {df_liv.shape}')
print()
nulls_cmd = df_cmd.isnull().sum()[df_cmd.isnull().sum() > 0]
nulls_liv = df_liv.isnull().sum()[df_liv.isnull().sum() > 0]
print('Nulls CMD restants :', nulls_cmd.to_dict() if len(nulls_cmd) > 0 else 'AUCUN')
print('Nulls LIV restants :', nulls_liv.to_dict() if len(nulls_liv) > 0 else 'AUCUN')

=== RESULTAT ETAPE 2 ===
CMD : (352518, 14)
LIV : (353295, 13)

Nulls CMD restants : AUCUN
Nulls LIV restants : AUCUN


In [19]:
out_dir = os.path.join(BASE_DIR, 'data', 'processed')
df_cmd.to_csv(os.path.join(out_dir, 'cmd_step2.csv'), index=False, encoding='utf-8')
df_liv.to_csv(os.path.join(out_dir, 'liv_step2.csv'), index=False, encoding='utf-8')
print('Sauvegarde : cmd_step2.csv, liv_step2.csv')

Sauvegarde : cmd_step2.csv, liv_step2.csv


---
## Étape 2 — TERMINÉE

| Colonne | Null avant | Null apres | Methode |
|---|---|---|---|
| `code_article` | 31 / 30 | 0 / 0 | Suppression des lignes null uniquement |
| `segment` | 19 / 19 | 0 / 0 | Resolu via suppression code_article (100% overlap) |
| `pays` | 99 / 100 | 0 / 0 | Mode par code_client, sinon UNKNOWN |
| `devise` | ~1987 / ~1977 | 0 / 0 | Mode par pays, sinon EUR |
| `famille_activite_client` | 17 / 17 | 0 / 0 | Mode par code_client, sinon AUTRE |

Lignes finales : **352 518 CMD** / **353 295 LIV**

**Prochaine etape (Lun 14 Avr) : Etape 3 — Normalisation des dates**

---
# Étape 3 — Normalisation des Dates

| Colonne | Format brut | Action |
|---|---|---|
| `date_enregistrement` | `dd/mm/yyyy` | `pd.to_datetime` format `%d/%m/%Y` |
| `date_livraison_demandee` | `dd/mm/yyyy` | idem |
| `date_livraison_reelle` | `dd/mm/yyyy` | idem |

## 3.1 — Conversion des dates

In [20]:
df_cmd = pd.read_csv(os.path.join(BASE_DIR, 'data/processed/cmd_step2.csv'), encoding='utf-8')
df_liv = pd.read_csv(os.path.join(BASE_DIR, 'data/processed/liv_step2.csv'), encoding='utf-8')

df_cmd['date_enregistrement']     = pd.to_datetime(df_cmd['date_enregistrement'],     format='%d/%m/%Y', errors='coerce')
df_cmd['date_livraison_demandee'] = pd.to_datetime(df_cmd['date_livraison_demandee'], format='%d/%m/%Y', errors='coerce')
df_liv['date_livraison_reelle']   = pd.to_datetime(df_liv['date_livraison_reelle'],   format='%d/%m/%Y', errors='coerce')

print('NaT après conversion :')
print(f'  date_enregistrement     : {df_cmd["date_enregistrement"].isna().sum()}')
print(f'  date_livraison_demandee : {df_cmd["date_livraison_demandee"].isna().sum()}')
print(f'  date_livraison_reelle   : {df_liv["date_livraison_reelle"].isna().sum()}')
print()
print('Plage CMD :', df_cmd['date_enregistrement'].min().date(), '->', df_cmd['date_enregistrement'].max().date())
print('Plage LIV :', df_liv['date_livraison_reelle'].min().date(), '->', df_liv['date_livraison_reelle'].max().date())

NaT après conversion :
  date_enregistrement     : 0
  date_livraison_demandee : 0
  date_livraison_reelle   : 0

Plage CMD : 2020-09-23 -> 2025-12-23
Plage LIV : 2021-01-04 -> 2025-12-23


## 3.2 — Distribution par année

In [21]:
print('=== Commandes par année (date_enregistrement) ===')
print(df_cmd['date_enregistrement'].dt.year.value_counts().sort_index().to_string())
print()
print('=== Livraisons par année (date_livraison_reelle) ===')
print(df_liv['date_livraison_reelle'].dt.year.value_counts().sort_index().to_string())

=== Commandes par année (date_enregistrement) ===
date_enregistrement
2020     1718
2021    75683
2022    67569
2023    69129
2024    71637
2025    66782

=== Livraisons par année (date_livraison_reelle) ===
date_livraison_reelle
2021    76161
2022    66548
2023    70078
2024    72004
2025    68504


## 3.3 — Detection et suppression des anomalies temporelles

**Anomalie detectee :** 3 lignes ou   
Commande 152881 : livree juillet 2021, enregistree septembre 2021 (saisie tardive).  
**Decision : suppression** —  serait incoherent et polluerait le modele. Perte : 3 / 353 295 = 0,0008%.

In [22]:
# Jointure pour detecter anomalies date_livraison_reelle < date_enregistrement
check = df_cmd[["num_commande","code_article","date_enregistrement"]].merge(
    df_liv[["num_commande","code_article","date_livraison_reelle"]],
    on=["num_commande","code_article"], how="inner"
)
anomalies = check[check["date_livraison_reelle"] < check["date_enregistrement"]]
print(f"Livraisons avant enregistrement : {len(anomalies)}")
print(anomalies.to_string())

# Supprimer ces lignes dans df_liv
n_avant = len(df_liv)
cles_anomalie = set(zip(anomalies["num_commande"], anomalies["code_article"]))
mask = df_liv.apply(lambda r: (r["num_commande"], r["code_article"]) in cles_anomalie, axis=1)
df_liv = df_liv[~mask]
print(f"LIV : {n_avant:,} -> {len(df_liv):,} ({n_avant - len(df_liv)} lignes supprimees)")


Livraisons avant enregistrement : 3
       num_commande code_article date_enregistrement date_livraison_reelle
45170        152881        A0858          2021-09-01            2021-07-02
45171        152881        A0989          2021-09-01            2021-07-02
45173        152881        A0592          2021-09-01            2021-07-02
LIV : 353,295 -> 353,292 (3 lignes supprimees)


## 3.4 — Sauvegarde step3

In [23]:
out_dir = os.path.join(BASE_DIR, 'data', 'processed')
df_cmd.to_csv(os.path.join(out_dir, 'cmd_step3.csv'), index=False, encoding='utf-8')
df_liv.to_csv(os.path.join(out_dir, 'liv_step3.csv'), index=False, encoding='utf-8')
print('Sauvegarde : cmd_step3.csv, liv_step3.csv')

Sauvegarde : cmd_step3.csv, liv_step3.csv


---
## Etape 3 — TERMINEE

| Verification | Resultat |
|---|---|
| NaT apres parsing | 0 / 0 / 0 |
| Dates hors plage 2020-2025 | 0 |
| Anomalies date_livraison_reelle < date_enregistrement | 3 lignes supprimees (commande 152881) |
| Plage CMD | 2020-09-23 -> 2025-12-23 |
| Plage LIV | 2021-01-04 -> 2025-12-23 |
| LIV apres step3 | 353 292 lignes |

**Prochaine etape (Mar 15 Avr) : Etape 4 — Jointure commandes x livraisons**

---
# Etape 4 - Jointure Commandes x Livraisons

**Strategie :**
- CMD : 2 790 paires `num_commande x code_article` dupliquees -> `sum(qte_demandee)`, `max(prix)`
- LIV : 3 779 paires avec livraisons fractionnees -> `sum(qte_livree)`, `max(date_livraison_reelle)`
- Jointure `LEFT` pour conserver les commandes non encore livrees


## 4.1 - Agregation avant jointure

In [24]:
df_cmd = pd.read_csv(os.path.join(BASE_DIR, 'data/processed/cmd_step3.csv'), encoding='utf-8',
                     parse_dates=['date_enregistrement','date_livraison_demandee'])
df_liv = pd.read_csv(os.path.join(BASE_DIR, 'data/processed/liv_step3.csv'), encoding='utf-8',
                     parse_dates=['date_livraison_reelle'])

cols_meta = ['num_commande','code_article','date_enregistrement','date_livraison_demandee',
             'code_client','famille_activite_client','famille_activite_article',
             'segment','type_activite','pays','devise']

df_cmd_agg = df_cmd.groupby(cols_meta, as_index=False).agg(
    qte_demandee=('qte_demandee', 'sum'),
    prix=('prix', 'max')
)
df_liv_agg = df_liv.groupby(['num_commande','code_article'], as_index=False).agg(
    qte_livree=('qte_livree', 'sum'),
    date_livraison_reelle=('date_livraison_reelle', 'max')
)

print(f'CMD apres agregation : {df_cmd_agg.shape}  (supprime {df_cmd.shape[0]-df_cmd_agg.shape[0]} doublons)')
print(f'LIV apres agregation : {df_liv_agg.shape}  (supprime {df_liv.shape[0]-df_liv_agg.shape[0]} doublons)')


CMD apres agregation : (349390, 13)  (supprime 3128 doublons)
LIV apres agregation : (349091, 4)  (supprime 4201 doublons)


## 4.2 - Jointure LEFT

In [25]:
df_merged = pd.merge(
    df_cmd_agg,
    df_liv_agg,
    on=['num_commande', 'code_article'],
    how='left'
)
print('Shape apres jointure :', df_merged.shape)
print('Commandes livrees    :', df_merged['qte_livree'].notna().sum())
print('Commandes en attente :', df_merged['qte_livree'].isna().sum())


Shape apres jointure : (349390, 15)
Commandes livrees    : 349058
Commandes en attente : 332


## 4.3 - Calcul statut, taux_satisfaction, delai_jours

In [26]:
# Statut
df_merged['statut'] = 'livre'
df_merged.loc[df_merged['qte_livree'].isna(), 'statut'] = 'en_attente'
df_merged['qte_livree'] = df_merged['qte_livree'].fillna(0)

# Taux de satisfaction
df_merged['taux_satisfaction'] = (df_merged['qte_livree'] / df_merged['qte_demandee']).clip(0, 1)

# Livraison excedentaire
df_merged['livraison_excedentaire'] = df_merged['qte_livree'] > df_merged['qte_demandee']

# Delai de livraison (positif=retard, negatif=avance)
df_merged['delai_jours'] = (df_merged['date_livraison_reelle'] - df_merged['date_livraison_demandee']).dt.days

print('taux_satisfaction — min:', round(df_merged['taux_satisfaction'].min(), 3),
      '  max:', round(df_merged['taux_satisfaction'].max(), 3),
      '  mean:', round(df_merged['taux_satisfaction'].mean(), 3))
print('delai_jours       — min:', df_merged['delai_jours'].min(),
      '  max:', df_merged['delai_jours'].max(),
      '  median:', df_merged['delai_jours'].median())
print('Livraisons excedentaires :', df_merged['livraison_excedentaire'].sum())


taux_satisfaction — min: 0.0   max: 1.0   mean: 0.999
delai_jours       — min: -124.0   max: 139.0   median: 0.0
Livraisons excedentaires : 55


## 4.4 - Verification et export

In [27]:
print('Shape final :', df_merged.shape)
print()
print('Statut :')
print(df_merged['statut'].value_counts().to_string())
print()
nulls = df_merged.isnull().sum()[df_merged.isnull().sum() > 0]
print('Nulls restants (date_livraison_reelle + delai_jours = commandes en_attente) :')
print(nulls.to_string())


Shape final : (349390, 19)

Statut :


statut
livre         349058
en_attente       332

Nulls restants (date_livraison_reelle + delai_jours = commandes en_attente) :
date_livraison_reelle    332
delai_jours              332


In [28]:
out_dir = os.path.join(BASE_DIR, 'data', 'processed')
df_merged.to_csv(os.path.join(out_dir, 'dataset_step4.csv'), index=False, encoding='utf-8')
df_merged.to_parquet(os.path.join(out_dir, 'dataset_step4.parquet'), index=False)
print('Sauvegarde : dataset_step4.csv + dataset_step4.parquet')
print('Colonnes :', list(df_merged.columns))


Sauvegarde : dataset_step4.csv + dataset_step4.parquet
Colonnes : ['num_commande', 'code_article', 'date_enregistrement', 'date_livraison_demandee', 'code_client', 'famille_activite_client', 'famille_activite_article', 'segment', 'type_activite', 'pays', 'devise', 'qte_demandee', 'prix', 'qte_livree', 'date_livraison_reelle', 'statut', 'taux_satisfaction', 'livraison_excedentaire', 'delai_jours']


In [29]:
df_merged['statut'].unique()

<ArrowStringArray>
['livre', 'en_attente']
Length: 2, dtype: str

---
## Etape 4 - TERMINEE

| Resultat | Valeur |
|---|---|
| Shape final | 349 390 lignes x 19 colonnes |
| Commandes livrees | 349 058 (99,9%) |
| Commandes en attente | 332 (0,1%) |
| taux_satisfaction moyen | 0.999 |
| delai_jours median | 0 jour |
| delai_jours plage | -124 a +139 jours |
| Livraisons excedentaires | 55 |

**Prochaine etape (Mer 16 Avr) : Etape 5 - Feature Engineering Temporel**


---
# Étape 5 — Feature Engineering Temporel

**Objectif :** Extraire des variables prédictives à partir des dates pour capturer la saisonnalité, le délai demandé, et le positionnement calendaire.

| Feature | Source | Description |
|---|---|---|
| `annee_cmd`, `mois_cmd`, `trimestre_cmd`, `jour_semaine_cmd` | `date_enregistrement` | Saisonnalité commande |
| `semaine_cmd` | `date_enregistrement` | Semaine ISO |
| `est_fin_mois_cmd` | `date_enregistrement` | Jour >= 25 (pic fin de mois) |
| `annee_liv_dem`, `mois_liv_dem`, `trimestre_liv_dem` | `date_livraison_demandee` | Saisonnalité livraison demandée |
| `jour_semaine_liv_dem`, `est_weekend_liv_dem` | `date_livraison_demandee` | Livraison demandée un weekend |
| `delai_demande_jours` | `date_liv_dem - date_enreg` | Délai client demandé (en jours) |
| `en_retard` | `delai_jours > 0` | **Variable cible** (classification binaire) |

## 5.1 — Chargement du dataset step4

In [30]:
import pandas as pd
import numpy as np
import os

BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()

df = pd.read_parquet(os.path.join(BASE_DIR, 'data/processed/dataset_step4.parquet'))
print(f'Shape chargé : {df.shape}')
print(f'Colonnes     : {list(df.columns)}')
print(f'Types dates  : date_enregistrement={df["date_enregistrement"].dtype}, date_livraison_demandee={df["date_livraison_demandee"].dtype}')

Shape chargé : (349390, 19)
Colonnes     : ['num_commande', 'code_article', 'date_enregistrement', 'date_livraison_demandee', 'code_client', 'famille_activite_client', 'famille_activite_article', 'segment', 'type_activite', 'pays', 'devise', 'qte_demandee', 'prix', 'qte_livree', 'date_livraison_reelle', 'statut', 'taux_satisfaction', 'livraison_excedentaire', 'delai_jours']
Types dates  : date_enregistrement=datetime64[us], date_livraison_demandee=datetime64[us]


## 5.2 — Features temporelles issues de date_enregistrement

In [31]:
# --- date_enregistrement ---
df['annee_cmd']        = df['date_enregistrement'].dt.year
df['mois_cmd']         = df['date_enregistrement'].dt.month          # 1-12
df['trimestre_cmd']    = df['date_enregistrement'].dt.quarter         # 1-4
df['semaine_cmd']      = df['date_enregistrement'].dt.isocalendar().week.astype(int)  # 1-53
df['jour_semaine_cmd'] = df['date_enregistrement'].dt.dayofweek      # 0=Lun, 6=Dim
df['est_fin_mois_cmd'] = (df['date_enregistrement'].dt.day >= 25).astype(int)

print('Features date_enregistrement créées :')
print(df[['date_enregistrement','annee_cmd','mois_cmd','trimestre_cmd','semaine_cmd','jour_semaine_cmd','est_fin_mois_cmd']].head(3).to_string())

Features date_enregistrement créées :
  date_enregistrement  annee_cmd  mois_cmd  trimestre_cmd  semaine_cmd  jour_semaine_cmd  est_fin_mois_cmd
0          2020-09-23       2020         9              3           39                 2                 0
1          2020-10-08       2020        10              4           41                 3                 0
2          2020-10-08       2020        10              4           41                 3                 0


## 5.3 — Features temporelles issues de date_livraison_demandee

In [32]:
# --- date_livraison_demandee ---
df['annee_liv_dem']        = df['date_livraison_demandee'].dt.year
df['mois_liv_dem']         = df['date_livraison_demandee'].dt.month
df['trimestre_liv_dem']    = df['date_livraison_demandee'].dt.quarter
df['jour_semaine_liv_dem'] = df['date_livraison_demandee'].dt.dayofweek   # 0=Lun, 6=Dim
df['est_weekend_liv_dem']  = (df['jour_semaine_liv_dem'] >= 5).astype(int)  # Sam=5, Dim=6

print('Features date_livraison_demandee créées :')
print(df[['date_livraison_demandee','annee_liv_dem','mois_liv_dem','trimestre_liv_dem','jour_semaine_liv_dem','est_weekend_liv_dem']].head(3).to_string())
print()
print('Livraisons demandées un weekend :', df['est_weekend_liv_dem'].sum(), f"({df['est_weekend_liv_dem'].mean()*100:.1f}%)")

Features date_livraison_demandee créées :
  date_livraison_demandee  annee_liv_dem  mois_liv_dem  trimestre_liv_dem  jour_semaine_liv_dem  est_weekend_liv_dem
0              2021-01-15           2021             1                  1                     4                    0
1              2021-01-04           2021             1                  1                     0                    0
2              2021-01-04           2021             1                  1                     0                    0

Livraisons demandées un weekend : 504 (0.1%)


## 5.4 — Délai demandé et variable cible

In [33]:
# Délai demandé par le client : date_livraison_demandee - date_enregistrement
df['delai_demande_jours'] = (df['date_livraison_demandee'] - df['date_enregistrement']).dt.days

# Variable cible binaire : en retard si livraison réelle > livraison demandée
# NaN pour commandes en_attente (pas encore livrées)
df['en_retard'] = np.where(
    df['statut'] == 'en_attente',
    np.nan,
    (df['delai_jours'] > 0).astype(float)
)

print('delai_demande_jours :')
print(f'  min={df["delai_demande_jours"].min()}  max={df["delai_demande_jours"].max()}  median={df["delai_demande_jours"].median()}')
print(f'  Négatifs (livraison demandée avant enregistrement) : {(df["delai_demande_jours"] < 0).sum()}')
print()
print('Variable cible en_retard (commandes livrées uniquement) :')
print(df['en_retard'].value_counts(dropna=True).to_string())
print(f'  Taux de retard : {df["en_retard"].mean()*100:.2f}%')

delai_demande_jours :
  min=-61  max=388  median=5.0
  Négatifs (livraison demandée avant enregistrement) : 96

Variable cible en_retard (commandes livrées uniquement) :
en_retard
0.0    281975
1.0     67083
  Taux de retard : 19.22%


## 5.5 — Vérification et distribution des features

In [34]:
new_features = [
    'annee_cmd','mois_cmd','trimestre_cmd','semaine_cmd','jour_semaine_cmd','est_fin_mois_cmd',
    'annee_liv_dem','mois_liv_dem','trimestre_liv_dem','jour_semaine_liv_dem','est_weekend_liv_dem',
    'delai_demande_jours','en_retard'
]

print(f'Shape après feature engineering : {df.shape}')
print(f'Nouvelles colonnes : {len(new_features)}')
print()
print('Nulls dans les nouvelles features (hors en_retard) :')
nulls = df[new_features].isnull().sum()
print(nulls[nulls > 0].to_string() if nulls[nulls > 0].any() else '  Aucun null (sauf en_retard=332 commandes en_attente)')
print()
print('Distribution mois_cmd :')
print(df['mois_cmd'].value_counts().sort_index().to_string())
print()
print('Distribution trimestre_cmd :')
print(df['trimestre_cmd'].value_counts().sort_index().to_string())

Shape après feature engineering : (349390, 32)
Nouvelles colonnes : 13

Nulls dans les nouvelles features (hors en_retard) :
en_retard    332

Distribution mois_cmd :
mois_cmd
1     37607
2     36388
3     33056
4     29976
5     26605
6     29102
7     26083
8     15944
9     29927
10    32325
11    28372
12    24005

Distribution trimestre_cmd :
trimestre_cmd
1    107051
2     85683
3     71954
4     84702


## 5.6 — Sauvegarde dataset_step5

In [35]:
out_dir = os.path.join(BASE_DIR, 'data', 'processed')
df.to_csv(os.path.join(out_dir, 'dataset_step5.csv'), index=False, encoding='utf-8')
df.to_parquet(os.path.join(out_dir, 'dataset_step5.parquet'), index=False)
print('Sauvegarde : dataset_step5.csv + dataset_step5.parquet')
print(f'Shape final : {df.shape}')
print(f'Colonnes ({len(df.columns)}) : {list(df.columns)}')

Sauvegarde : dataset_step5.csv + dataset_step5.parquet
Shape final : (349390, 32)
Colonnes (32) : ['num_commande', 'code_article', 'date_enregistrement', 'date_livraison_demandee', 'code_client', 'famille_activite_client', 'famille_activite_article', 'segment', 'type_activite', 'pays', 'devise', 'qte_demandee', 'prix', 'qte_livree', 'date_livraison_reelle', 'statut', 'taux_satisfaction', 'livraison_excedentaire', 'delai_jours', 'annee_cmd', 'mois_cmd', 'trimestre_cmd', 'semaine_cmd', 'jour_semaine_cmd', 'est_fin_mois_cmd', 'annee_liv_dem', 'mois_liv_dem', 'trimestre_liv_dem', 'jour_semaine_liv_dem', 'est_weekend_liv_dem', 'delai_demande_jours', 'en_retard']


In [37]:
pd.set_option('display.max_columns', None)
df.head()

,num_commande,code_article,date_enregistrement,date_livraison_demandee,code_client,famille_activite_client,famille_activite_article,segment,type_activite,pays,devise,qte_demandee,prix,qte_livree,date_livraison_reelle,statut,taux_satisfaction,livraison_excedentaire,delai_jours,annee_cmd,mois_cmd,trimestre_cmd,semaine_cmd,jour_semaine_cmd,est_fin_mois_cmd,annee_liv_dem,mois_liv_dem,trimestre_liv_dem,jour_semaine_liv_dem,est_weekend_liv_dem,delai_demande_jours,en_retard
0,142660,A0276,2020-09-23,2021-01-15,C10379,ELEVAGE,ELEVAGE,ECORNAGE,Outil GAZ,IE,EUR,300,89.00,300.0,2021-02-17,livre,1.0,False,33.0,2020,9,3,39,2,0,2021,1,1,4,0,114,1.0
1,143298,A0441,2020-10-08,2021-01-04,C10567,CONSTRUCTION,CONSTRUCTION,OUTIL CHAUFFANT,GAZ,FR,EUR,12,6.77,12.0,2021-01-04,livre,1.0,False,0.0,2020,10,4,41,3,0,2021,1,1,0,0,88,0.0
2,143298,A0577,2020-10-08,2021-01-04,C10567,CONSTRUCTION,CONSTRUCTION,COUVERTURE,PRESENTOIR,FR,EUR,1,0.00,1.0,2021-01-04,livre,1.0,False,0.0,2020,10,4,41,3,0,2021,1,1,0,0,88,0.0
3,143298,A0829,2020-10-08,2021-01-04,C10567,CONSTRUCTION,CONSTRUCTION,OUTIL CHAUFFANT,ACCESSOIRE,FR,EUR,1,19.22,1.0,2021-01-04,livre,1.0,False,0.0,2020,10,4,41,3,0,2021,1,1,0,0,88,0.0
4,143298,A0989,2020-10-08,2021-01-04,C10567,CONSTRUCTION,CONSTRUCTION,OUTIL CHAUFFANT,ACCESSOIRE,FR,EUR,3,16.33,3.0,2021-01-04,livre,1.0,False,0.0,2020,10,4,41,3,0,2021,1,1,0,0,88,0.0


---
## Étape 5 — TERMINÉE

| Feature | Type | Valeurs |
|---|---|---|
| `annee_cmd` | int | 2020–2025 |
| `mois_cmd` | int | 1–12 |
| `trimestre_cmd` | int | 1–4 |
| `semaine_cmd` | int | 1–53 |
| `jour_semaine_cmd` | int | 0=Lun … 6=Dim |
| `est_fin_mois_cmd` | 0/1 | jour >= 25 |
| `annee_liv_dem` | int | 2021–2025 |
| `mois_liv_dem` | int | 1–12 |
| `trimestre_liv_dem` | int | 1–4 |
| `jour_semaine_liv_dem` | int | 0=Lun … 6=Dim |
| `est_weekend_liv_dem` | 0/1 | Sam ou Dim |
| `delai_demande_jours` | int | délai demandé par le client |
| `en_retard` | 0/1/NaN | **cible** — NaN si en_attente |

Shape : **349 390 lignes × 32 colonnes**

**Prochaine étape : Étape 6 — Encodage des variables catégorielles**

---
# Étape 6 — Encodage des Variables Catégorielles

**Stratégie :**
- **Binaire direct** : `statut` (livre=1, en_attente=0)
- **Label Encoding** : colonnes à faible cardinalité (`devise`, `pays`, `famille_activite_client`, `famille_activite_article`, `segment`, `type_activite`)
- **Frequency Encoding** : `code_client`, `code_article` (haute cardinalité — remplacer par le nombre d'occurrences dans le dataset)

Les colonnes originales sont conservées. Les versions encodées portent le suffixe `_enc` ou `_freq`.

## 6.1 — Chargement du dataset step5

In [43]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder

BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()

df = pd.read_parquet(os.path.join(BASE_DIR, 'data/processed/dataset_step5.parquet'))
print(f'Shape chargé : {df.shape}')

# Cardinalité des colonnes catégorielles
cat_cols = ['statut', 'devise', 'pays', 'famille_activite_client',
            'famille_activite_article', 'segment', 'type_activite',
            'code_client', 'code_article']
print('\nCardinalités :')
for col in cat_cols:
    print(f'  {col:<30} : {df[col].nunique():>6} valeurs uniques')

Shape chargé : (349390, 32)

Cardinalités :
  statut                         :      2 valeurs uniques
  devise                         :      3 valeurs uniques
  pays                           :     79 valeurs uniques
  famille_activite_client        :      4 valeurs uniques
  famille_activite_article       :      3 valeurs uniques
  segment                        :      9 valeurs uniques
  type_activite                  :     12 valeurs uniques
  code_client                    :   3465 valeurs uniques
  code_article                   :   1068 valeurs uniques


## 6.2 — Encodage binaire : statut

In [44]:
# statut : livre=1, en_attente=0
df['statut_enc'] = (df['statut'] == 'livre').astype(int)

print('statut_enc :')
print(df.groupby('statut')['statut_enc'].first().to_string())

statut_enc :
statut
en_attente    0
livre         1


In [55]:
df.head()

,num_commande,code_article,date_enregistrement,date_livraison_demandee,code_client,famille_activite_client,famille_activite_article,segment,type_activite,pays,devise,qte_demandee,prix,qte_livree,date_livraison_reelle,statut,taux_satisfaction,livraison_excedentaire,delai_jours,annee_cmd,mois_cmd,trimestre_cmd,semaine_cmd,jour_semaine_cmd,est_fin_mois_cmd,annee_liv_dem,mois_liv_dem,trimestre_liv_dem,jour_semaine_liv_dem,est_weekend_liv_dem,delai_demande_jours,en_retard,statut_enc,devise_enc,pays_enc,famille_activite_client_enc,famille_activite_article_enc,segment_enc,type_activite_enc,code_client_freq,code_article_freq
0,142660,A0276,2020-09-23,2021-01-15,C10379,ELEVAGE,ELEVAGE,ECORNAGE,Outil GAZ,IE,EUR,300,89.00,300.0,2021-02-17,livre,1.0,False,33.0,2020,9,3,39,2,0,2021,1,1,4,0,114,1.0,1,1,37,2,1,4,7,296,115
1,143298,A0441,2020-10-08,2021-01-04,C10567,CONSTRUCTION,CONSTRUCTION,OUTIL CHAUFFANT,GAZ,FR,EUR,12,6.77,12.0,2021-01-04,livre,1.0,False,0.0,2020,10,4,41,3,0,2021,1,1,0,0,88,0.0,1,1,27,1,0,6,4,159,6253
2,143298,A0577,2020-10-08,2021-01-04,C10567,CONSTRUCTION,CONSTRUCTION,COUVERTURE,PRESENTOIR,FR,EUR,1,0.00,1.0,2021-01-04,livre,1.0,False,0.0,2020,10,4,41,3,0,2021,1,1,0,0,88,0.0,1,1,27,1,0,3,10,159,24
3,143298,A0829,2020-10-08,2021-01-04,C10567,CONSTRUCTION,CONSTRUCTION,OUTIL CHAUFFANT,ACCESSOIRE,FR,EUR,1,19.22,1.0,2021-01-04,livre,1.0,False,0.0,2020,10,4,41,3,0,2021,1,1,0,0,88,0.0,1,1,27,1,0,6,0,159,782
4,143298,A0989,2020-10-08,2021-01-04,C10567,CONSTRUCTION,CONSTRUCTION,OUTIL CHAUFFANT,ACCESSOIRE,FR,EUR,3,16.33,3.0,2021-01-04,livre,1.0,False,0.0,2020,10,4,41,3,0,2021,1,1,0,0,88,0.0,1,1,27,1,0,6,0,159,1632


## 6.3 — Label Encoding : colonnes à faible cardinalité

In [46]:
from sklearn.preprocessing import LabelEncoder

label_cols = [
    'devise',
    'pays',
    'famille_activite_client',
    'famille_activite_article',
    'segment',
    'type_activite',
]

label_encoders = {}
for col in label_cols:
    le = LabelEncoder()
    df[f'{col}_enc'] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f'{col:<30} : {df[col].nunique()} classes  -> {col}_enc  (ex: {dict(zip(le.classes_[:3], le.transform(le.classes_[:3])))})')

devise                         : 3 classes  -> devise_enc  (ex: {'CNY': np.int64(0), 'EUR': np.int64(1), 'USD': np.int64(2)})
pays                           : 79 classes  -> pays_enc  (ex: {'AD': np.int64(0), 'AE': np.int64(1), 'AR': np.int64(2)})
famille_activite_client        : 4 classes  -> famille_activite_client_enc  (ex: {'AUTRE': np.int64(0), 'CONSTRUCTION': np.int64(1), 'ELEVAGE': np.int64(2)})
famille_activite_article       : 3 classes  -> famille_activite_article_enc  (ex: {'CONSTRUCTION': np.int64(0), 'ELEVAGE': np.int64(1), 'RETRACTION': np.int64(2)})
segment                        : 9 classes  -> segment_enc  (ex: {'BITUME': np.int64(0), 'CONNECTIQUE': np.int64(1), 'CONTENTION': np.int64(2)})
type_activite                  : 12 classes  -> type_activite_enc  (ex: {'ACCESSOIRE': np.int64(0), 'CONSOMMABLE': np.int64(1), 'CONSOMMABLE BRASURE': np.int64(2)})


## 6.4 — Frequency Encoding : code_client et code_article

Pour les identifiants à haute cardinalité, on remplace chaque valeur par son **nombre d'occurrences** dans le dataset.
Cela donne une information utile : un client fréquent est différent d'un client occasionnel.

In [47]:
for col in ['code_client', 'code_article']:
    freq = df[col].value_counts()
    df[f'{col}_freq'] = df[col].map(freq)
    print(f'{col}_freq — min={df[f"{col}_freq"].min()}  max={df[f"{col}_freq"].max()}  median={df[f"{col}_freq"].median()}')

print()
print('Top 5 clients (occurrences dans le dataset) :')
print(df.groupby('code_client')['code_client_freq'].first().sort_values(ascending=False).head(5).to_string())

code_client_freq — min=1  max=4578  median=365.0
code_article_freq — min=1  max=11553  median=2573.0

Top 5 clients (occurrences dans le dataset) :
code_client
C12289    4578
C10917    3964
C05933    3909
C12408    3617
C03917    3540


## 6.5 — Vérification finale

In [48]:
enc_cols = ['statut_enc'] + [f'{c}_enc' for c in label_cols] + ['code_client_freq', 'code_article_freq']

print(f'Shape : {df.shape}')
print(f'Nouvelles colonnes encodées ({len(enc_cols)}) : {enc_cols}')
print()
print('Nulls dans les colonnes encodées :')
nulls = df[enc_cols].isnull().sum()
print(nulls[nulls > 0].to_string() if nulls[nulls > 0].any() else '  Aucun null')
print()
print('Types des nouvelles colonnes :')
print(df[enc_cols].dtypes.to_string())

Shape : (349390, 41)
Nouvelles colonnes encodées (9) : ['statut_enc', 'devise_enc', 'pays_enc', 'famille_activite_client_enc', 'famille_activite_article_enc', 'segment_enc', 'type_activite_enc', 'code_client_freq', 'code_article_freq']

Nulls dans les colonnes encodées :
  Aucun null

Types des nouvelles colonnes :
statut_enc                      int64
devise_enc                      int64
pays_enc                        int64
famille_activite_client_enc     int64
famille_activite_article_enc    int64
segment_enc                     int64
type_activite_enc               int64
code_client_freq                int64
code_article_freq               int64


## 6.6 — Sauvegarde dataset_step6

In [49]:
out_dir = os.path.join(BASE_DIR, 'data', 'processed')
df.to_parquet(os.path.join(out_dir, 'dataset_step6.parquet'), index=False)
df.to_csv(os.path.join(out_dir, 'dataset_step6.csv'), index=False, encoding='utf-8')
print('Sauvegarde : dataset_step6.parquet + dataset_step6.csv')
print(f'Shape final : {df.shape}')
print(f'Colonnes ({len(df.columns)}) : {list(df.columns)}')

Sauvegarde : dataset_step6.parquet + dataset_step6.csv
Shape final : (349390, 41)
Colonnes (41) : ['num_commande', 'code_article', 'date_enregistrement', 'date_livraison_demandee', 'code_client', 'famille_activite_client', 'famille_activite_article', 'segment', 'type_activite', 'pays', 'devise', 'qte_demandee', 'prix', 'qte_livree', 'date_livraison_reelle', 'statut', 'taux_satisfaction', 'livraison_excedentaire', 'delai_jours', 'annee_cmd', 'mois_cmd', 'trimestre_cmd', 'semaine_cmd', 'jour_semaine_cmd', 'est_fin_mois_cmd', 'annee_liv_dem', 'mois_liv_dem', 'trimestre_liv_dem', 'jour_semaine_liv_dem', 'est_weekend_liv_dem', 'delai_demande_jours', 'en_retard', 'statut_enc', 'devise_enc', 'pays_enc', 'famille_activite_client_enc', 'famille_activite_article_enc', 'segment_enc', 'type_activite_enc', 'code_client_freq', 'code_article_freq']


---
## Étape 6 — TERMINÉE

| Colonne | Méthode | Nouvelle colonne |
|---|---|---|
| `statut` | Binaire (livre=1) | `statut_enc` |
| `devise` | Label Encoding | `devise_enc` |
| `pays` | Label Encoding | `pays_enc` |
| `famille_activite_client` | Label Encoding | `famille_activite_client_enc` |
| `famille_activite_article` | Label Encoding | `famille_activite_article_enc` |
| `segment` | Label Encoding | `segment_enc` |
| `type_activite` | Label Encoding | `type_activite_enc` |
| `code_client` | Frequency Encoding | `code_client_freq` |
| `code_article` | Frequency Encoding | `code_article_freq` |

**Prochaine étape : Étape 7 — Vérification finale et export du dataset ML**

---
# Étape 7 — Vérification Finale et Export du Dataset ML

**Objectif :** Produire `dataset_ml_final.parquet` — le fichier d'entrée de la Phase 2 (modélisation).

Actions :
1. Charger `dataset_step6`
2. Sélectionner uniquement les colonnes utiles au modèle
3. Vérifier l'absence de nulls dans les features ML
4. Afficher un bilan complet
5. Exporter le dataset final

**Colonnes exclues du dataset ML :**
- Identifiants texte bruts : `num_commande`, `code_client`, `code_article` (remplacés par freq)
- Dates brutes : `date_enregistrement`, `date_livraison_demandee`, `date_livraison_reelle` (déjà décomposées)
- Colonnes catégorielles brutes : `statut`, `devise`, `pays`, etc. (remplacées par _enc)
- `delai_jours` : information post-livraison, **data leakage** pour la cible `en_retard`
- `taux_satisfaction`, `livraison_excedentaire` : idem, connues seulement après livraison

## 7.1 — Chargement du dataset step6

In [50]:
import pandas as pd
import numpy as np
import os

BASE_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()

df = pd.read_parquet(os.path.join(BASE_DIR, 'data/processed/dataset_step6.parquet'))
print(f'Shape chargé : {df.shape}')
print(f'Toutes les colonnes ({len(df.columns)}) :')
for c in df.columns:
    print(f'  {c}')

Shape chargé : (349390, 41)
Toutes les colonnes (41) :
  num_commande
  code_article
  date_enregistrement
  date_livraison_demandee
  code_client
  famille_activite_client
  famille_activite_article
  segment
  type_activite
  pays
  devise
  qte_demandee
  prix
  qte_livree
  date_livraison_reelle
  statut
  taux_satisfaction
  livraison_excedentaire
  delai_jours
  annee_cmd
  mois_cmd
  trimestre_cmd
  semaine_cmd
  jour_semaine_cmd
  est_fin_mois_cmd
  annee_liv_dem
  mois_liv_dem
  trimestre_liv_dem
  jour_semaine_liv_dem
  est_weekend_liv_dem
  delai_demande_jours
  en_retard
  statut_enc
  devise_enc
  pays_enc
  famille_activite_client_enc
  famille_activite_article_enc
  segment_enc
  type_activite_enc
  code_client_freq
  code_article_freq


## 7.2 — Sélection des features ML

**Colonnes exclues (data leakage ou redondantes) :**
- `delai_jours`, `taux_satisfaction`, `livraison_excedentaire` : connues **après** livraison → le modèle ne peut pas les utiliser pour prédire
- Dates brutes et identifiants texte : déjà décomposés ou encodés

In [51]:
# Colonnes features (X) — utilisables par le modèle au moment de la prédiction
FEATURE_COLS = [
    # Quantités et prix
    'qte_demandee', 'prix',
    # Features temporelles commande
    'annee_cmd', 'mois_cmd', 'trimestre_cmd', 'semaine_cmd',
    'jour_semaine_cmd', 'est_fin_mois_cmd',
    # Features temporelles livraison demandée
    'annee_liv_dem', 'mois_liv_dem', 'trimestre_liv_dem',
    'jour_semaine_liv_dem', 'est_weekend_liv_dem',
    # Délai accordé par le client
    'delai_demande_jours',
    # Catégorielles encodées
    'statut_enc', 'devise_enc', 'pays_enc',
    'famille_activite_client_enc', 'famille_activite_article_enc',
    'segment_enc', 'type_activite_enc',
    # Fréquences
    'code_client_freq', 'code_article_freq',
]

# Colonne cible (y)
TARGET_COL = 'en_retard'

# Dataset ML = features + cible
df_ml = df[FEATURE_COLS + [TARGET_COL]].copy()

print(f'Dataset ML : {df_ml.shape}')
print(f'Features   : {len(FEATURE_COLS)}')
print(f'Cible      : {TARGET_COL}')
print(f'\nLignes avec cible connue (livrées) : {df_ml[TARGET_COL].notna().sum():,}')
print(f'Lignes sans cible (en attente)     : {df_ml[TARGET_COL].isna().sum():,}')

Dataset ML : (349390, 24)
Features   : 23
Cible      : en_retard

Lignes avec cible connue (livrées) : 349,058
Lignes sans cible (en attente)     : 332


## 7.3 — Vérification finale : nulls et types

In [52]:
print('=== NULLS dans les features (X) ===')
nulls_feat = df_ml[FEATURE_COLS].isnull().sum()
print(nulls_feat[nulls_feat > 0].to_string() if nulls_feat[nulls_feat > 0].any() else '  Aucun null dans les features')

print()
print('=== TYPES ===')
print(df_ml.dtypes.to_string())

print()
print('=== DISTRIBUTION DE LA CIBLE ===')
counts = df_ml[TARGET_COL].value_counts(dropna=True)
print(f'  en_retard=0 (à temps) : {int(counts.get(0.0, 0)):>7,}  ({counts.get(0.0,0)/df_ml[TARGET_COL].notna().sum()*100:.1f}%)')
print(f'  en_retard=1 (retard)  : {int(counts.get(1.0, 0)):>7,}  ({counts.get(1.0,0)/df_ml[TARGET_COL].notna().sum()*100:.1f}%)')
print(f'  NaN (en attente)      : {df_ml[TARGET_COL].isna().sum():>7,}')

=== NULLS dans les features (X) ===
  Aucun null dans les features

=== TYPES ===
qte_demandee                      int64
prix                            float64
annee_cmd                         int32
mois_cmd                          int32
trimestre_cmd                     int32
semaine_cmd                       int64
jour_semaine_cmd                  int32
est_fin_mois_cmd                  int64
annee_liv_dem                     int32
mois_liv_dem                      int32
trimestre_liv_dem                 int32
jour_semaine_liv_dem              int32
est_weekend_liv_dem               int64
delai_demande_jours               int64
statut_enc                        int64
devise_enc                        int64
pays_enc                          int64
famille_activite_client_enc       int64
famille_activite_article_enc      int64
segment_enc                       int64
type_activite_enc                 int64
code_client_freq                  int64
code_article_freq                 int6

## 7.4 — Export du dataset ML final

In [53]:
out_dir = os.path.join(BASE_DIR, 'data', 'processed')

# Export principal : parquet (rapide, typé, léger)
df_ml.to_parquet(os.path.join(out_dir, 'dataset_ml_final.parquet'), index=False)

# Export CSV de référence
df_ml.to_csv(os.path.join(out_dir, 'dataset_ml_final.csv'), index=False, encoding='utf-8')

print('=== EXPORT TERMINÉ ===')
print(f'  dataset_ml_final.parquet')
print(f'  dataset_ml_final.csv')
print()
print(f'Shape   : {df_ml.shape[0]:,} lignes × {df_ml.shape[1]} colonnes')
print(f'Features: {len(FEATURE_COLS)}')
print(f'Cible   : {TARGET_COL}')
print()
print('Fichiers dans data/processed :')
for f in sorted(os.listdir(out_dir)):
    size_kb = os.path.getsize(os.path.join(out_dir, f)) // 1024
    print(f'  {f:<40} {size_kb:>7} KB')

=== EXPORT TERMINÉ ===
  dataset_ml_final.parquet
  dataset_ml_final.csv

Shape   : 349,390 lignes × 24 colonnes
Features: 23
Cible   : en_retard

Fichiers dans data/processed :
  cmd_step1.csv                              39144 KB
  cmd_step2.csv                              39147 KB
  cmd_step3.csv                              39147 KB
  dataset_ml_final.csv                       23247 KB
  dataset_ml_final.parquet                    2201 KB
  dataset_step4.csv                          48630 KB
  dataset_step4.parquet                       3409 KB
  dataset_step5.csv                          60696 KB
  dataset_step5.parquet                       3668 KB
  dataset_step6.csv                          68799 KB
  dataset_step6.parquet                       4693 KB
  liv_step1.csv                              35434 KB
  liv_step2.csv                              35438 KB
  liv_step3.csv                              35438 KB


---
## Étape 7 — TERMINÉE ✓ Phase 1 complète

### Bilan Phase 1 — Data Engineering

| Étape | Action | Résultat |
|---|---|---|
| 1 | Chargement + renommage | 352 549 CMD / 353 325 LIV |
| 2 | Valeurs manquantes | 0 null restants |
| 3 | Normalisation dates | 3 anomalies supprimées |
| 4 | Jointure CMD × LIV | 349 390 lignes × 19 col |
| 5 | Feature engineering temporel | +13 features, cible `en_retard` créée |
| 6 | Encodage catégorielles | +9 colonnes encodées |
| 7 | Vérification + export | `dataset_ml_final.parquet` prêt |